# 00 · Ingesta, Descarga y descompresión

- **Proyecto:** Detección de Residuos Urbanos · Computer Vision · UNI 2026-I
- **Autor:** Jairzinho Santos

Deja los datasets públicos del proyecto en dos capas:

| Capa | Contenido | Regla |
|---|---|---|
| `data/raw/` | ZIP y JSON tal como llegaron, con manifiesto y MD5 | **Inmutable.** Fuente re-descargable. Nunca se edita |
| `data/bronze/` | Descomprimido y navegable, estructura homogénea `<ds>/images/` + `<ds>/annotations/` | **Sin transformación semántica.** Reconstruible desde `raw` sin red |

Las capas `silver` (COCO unificado, taxonomía, dedup, splits congelados) y `gold`
(formatos listos para entrenar) se construyen en notebooks posteriores.

### Propiedades

- **Reanudable** - si se corta, se reejecuta la misma celda y continúa (`curl -C -`)
- **Paralelo** - descargas y extracciones simultáneas
- **Idempotente** - lo ya completo se omite; reejecutar es seguro
- **Verificado** - MD5 oficial de Zenodo, integridad de ZIP y validez de JSON
- **Sin suspensión** - `caffeinate` activo mientras dure el proceso

### Antes de ejecutar

1. Laptop por **cable Ethernet**, no WiFi.
2. **Cargador conectado** - `caffeinate` evita la suspensión, no el consumo de batería.
3. Volumen **JS-MAIN** montado.

> Ejecutar las celdas en orden. Las celdas 6 (descarga) y 10 (descompresión) son las largas
> y muestran progreso en vivo.

## 1 · Configuración

In [ ]:
from pathlib import Path
import os, sys, csv, json, time, shutil, hashlib, zipfile, subprocess, datetime
from concurrent.futures import ThreadPoolExecutor

# ── Rutas ────────────────────────────────────────────────────────────────────
_cwd        = Path.cwd()
CODE_DIR    = _cwd.parent if _cwd.name == "notebooks" else _cwd
DATA_RAW    = CODE_DIR / "data" / "raw"
DATA_BRONZE = CODE_DIR / "data" / "bronze"
LOGS_DIR    = CODE_DIR / "data" / "_logs"
MANIFEST    = DATA_RAW / "_manifest_descarga.json"

# ── Qué descargar ────────────────────────────────────────────────────────────
INCLUIR_TIER1 = True    # TACO · RoLID-11K · UAVVaste · anotaciones  (núcleo)
INCLUIR_TIER2 = True    # ZeroWaste-f · ZeroWaste-w  (dominio industrial, opcional)
INCLUIR_TIER3 = False   # ZeroWaste aug/s (~19 GB de variantes sintéticas)

# ── Rendimiento ──────────────────────────────────────────────────────────────
MAX_PARALELAS  = 4      # descargas simultáneas
MAX_EXTRACCION = 4      # descompresiones simultáneas
VERIFICAR_MD5  = True   # MD5 contra los checksums oficiales de Zenodo
REFRESCO_SEG   = 2.0    # segundos entre repintados de la tabla de progreso

# ── Control explícito de rehacer trabajo ─────────────────────────────────────
# Esta notebook NUNCA borra nada por su cuenta. Lo ya presente y válido se omite.
# Para rehacer algo hay que pedirlo aquí, nombrando el dataset:
#     FORZAR_REDESCARGA  = ["taco"]      -> borra raw/taco/ y lo vuelve a bajar
#     FORZAR_REEXTRACCION = ["zerowaste_f"]  -> borra bronze/zerowaste_f/ y reextrae
FORZAR_REDESCARGA   = []
FORZAR_REEXTRACCION = []

for d in (DATA_RAW, DATA_BRONZE, LOGS_DIR):
    d.mkdir(parents=True, exist_ok=True)


def _fmt(n):
    """Formatea bytes de forma legible."""
    if n is None:
        return "?"
    n = float(n)
    for u in ("B", "KB", "MB", "GB", "TB"):
        if abs(n) < 1024.0:
            return "{:,.1f} {}".format(n, u)
        n /= 1024.0
    return "{:,.1f} PB".format(n)


print("CODE_DIR    : {}".format(CODE_DIR))
print("DATA_RAW    : {}".format(DATA_RAW))
print("DATA_BRONZE : {}".format(DATA_BRONZE))
print("Python      : {}".format(sys.version.split()[0]))

## 2 · Registro de datasets

Declarativo: para añadir o quitar una fuente basta editar este diccionario.

`bytes` solo se declara cuando la fuente publica el tamaño exacto (Zenodo). Para los
archivos servidos por GitHub raw se deja en `None`: no hay tamaño de referencia fiable,
así que se vuelven a descargar siempre (son unas decenas de MB) en lugar de compararlos
contra una estimación.

In [ ]:
ZEN    = "https://zenodo.org/api/records/{rec}/files/{key}/content"
RAW_GH = "https://raw.githubusercontent.com/{repo}/{branch}/{path}"

DATASETS = {

    # ───────────────────────────── TIER 1 · núcleo ─────────────────────────────
    "taco": {
        "tier": 1,
        "nombre": "TACO — Trash Annotations in Context",
        "dominio": "Peatón / mano · exterior",
        "contenido": "1.500 imgs · 4.784 anotaciones · 60 clases · bbox + polígonos",
        "licencia": "CC BY 4.0 (imágenes) / MIT (repo)",
        "cita": "Proença & Simões, arXiv:2003.06975",
        "archivos": [
            {"nombre": "TACO.zip",
             "url": ZEN.format(rec=3587843, key="TACO.zip"),
             "bytes": 2719810615, "md5": "e9149407d883e21a8d224feef8210920"},
            {"nombre": "annotations.json",
             "url": RAW_GH.format(repo="pedropro/TACO", branch="master", path="data/annotations.json"),
             "bytes": None, "md5": None},
            {"nombre": "annotations_unofficial.json",
             "url": RAW_GH.format(repo="pedropro/TACO", branch="master", path="data/annotations_unofficial.json"),
             "bytes": None, "md5": None},
            {"nombre": "all_image_urls.csv",
             "url": RAW_GH.format(repo="pedropro/TACO", branch="master", path="data/all_image_urls.csv"),
             "bytes": None, "md5": None},
        ],
    },

    "rolid11k": {
        "tier": 1,
        "nombre": "RoLID-11K — Roadside Litter (dashcam)",
        "dominio": "Vehículo / dashcam · Reino Unido",
        "contenido": "11.565 imgs · 1 clase · split oficial · >80% objetos small",
        "licencia": "Apache-2.0",
        "cita": "Wu et al., WACV Workshops 2026",
        "modo": "gdown_folder",
        "folder_id": "1aPPW76js_JKdx_9SF0tioxjs17sLbOn-",
        # Permite omitir el re-escaneo de Drive (lento) cuando ya está todo bajado
        "esperado_archivos": ["0-11.zip", "12_15.zip", "15_18.zip", "18_21.zip", "21_24.zip",
                              "training.json", "validation.json", "testing.json"],
        "archivos": [],
    },

    "uavvaste": {
        "tier": 1,
        "nombre": "UAVVaste — litter aéreo (dron)",
        "dominio": "Dron / cenital baja altura · Polonia",
        "contenido": "772 imgs · 3.718 anotaciones · 1 clase · bbox + máscaras COCO",
        "licencia": "CC BY 4.0",
        "cita": "Kraft et al., Remote Sensing 13(5):965, 2021",
        "archivos": [
            {"nombre": "UAVVasteDataset.zip",
             "url": ZEN.format(rec=8214061, key="UAVVasteDataset.zip"),
             "bytes": 3010727259, "md5": "1575c32c04bdf944047563e4a1786c2a"},
        ],
    },

    "detectwaste_ann": {
        "tier": 1,
        "nombre": "detect-waste — anotaciones (Extended TACO)",
        "dominio": "Anotaciones corregidas · sin imágenes",
        "contenido": "TACO extendido con inconsistencias de etiqueta corregidas + índice OpenLitterMap",
        "licencia": "ver repo wimlds-trojmiasto/detect-waste",
        "cita": "Majchrowska et al., Waste Management 138, 2021",
        "archivos": [
            {"nombre": f,
             "url": RAW_GH.format(repo="wimlds-trojmiasto/detect-waste",
                                  branch="main", path="annotations/" + f),
             "bytes": None, "md5": None}
            for f in ("annotations_train.json", "annotations_test.json",
                      "annotations_binary_train.json", "annotations_binary_test.json",
                      "binary_mixed_train.json", "binary_mixed_test.json",
                      "openlittermap.json")
        ],
    },

    # ────────────────────── TIER 2 · opcional (domain shift) ──────────────────────
    "zerowaste_f": {
        "tier": 2,
        "nombre": "ZeroWaste-f — planta de reciclaje",
        "dominio": "Cinta transportadora industrial · interior",
        "contenido": "4.503 frames · 4 materiales · splits oficiales + máscaras semánticas",
        "licencia": "CC BY-NC 4.0",
        "cita": "Bashkirova et al., CVPR 2022",
        "archivos": [
            {"nombre": "zerowaste-f-final.zip",
             "url": ZEN.format(rec=6412647, key="zerowaste-f-final.zip"),
             "bytes": 7518242799, "md5": "e26e31a58080bca6782dca0e56074c5d"},
        ],
    },

    "zerowaste_w": {
        "tier": 2,
        "nombre": "ZeroWaste-w — frames sin etiquetar",
        "dominio": "Cinta transportadora industrial · interior",
        "contenido": "2.410 frames sin anotar (before/after)",
        "licencia": "CC BY-NC 4.0",
        "cita": "Bashkirova et al., CVPR 2022",
        "archivos": [
            {"nombre": "zerowaste-w.zip",
             "url": ZEN.format(rec=6412647, key="zerowaste-w.zip"),
             "bytes": 3267632268, "md5": "7c4d3f5c875a1f25e252b0d81f3b6c1b"},
        ],
    },

    # ─────────────── TIER 3 · variantes sintéticas (por defecto OFF) ───────────────
    "zerowaste_aug": {
        "tier": 3,
        "nombre": "ZeroWaste-aug — variante aumentada (multiparte)",
        "dominio": "Cinta transportadora · sintético",
        "contenido": "ZIP multiparte .z01-.z03 + .zip",
        "licencia": "CC BY-NC 4.0",
        "cita": "Bashkirova et al., CVPR 2022",
        "archivos": [
            {"nombre": "zerowaste-aug.z01", "url": ZEN.format(rec=6412647, key="zerowaste-aug.z01"),
             "bytes": 2147483648, "md5": "3c6f88809adb0fcd362f4f6c59543639"},
            {"nombre": "zerowaste-aug.z02", "url": ZEN.format(rec=6412647, key="zerowaste-aug.z02"),
             "bytes": 2147483648, "md5": "5d77f6c40a1ab9f92d2d4a2d664cbbf7"},
            {"nombre": "zerowaste-aug.z03", "url": ZEN.format(rec=6412647, key="zerowaste-aug.z03"),
             "bytes": 2147483648, "md5": "ee97ea4dccb6c027f39eda2d96f9f717"},
            {"nombre": "zerowaste-aug.zip", "url": ZEN.format(rec=6412647, key="zerowaste-aug.zip"),
             "bytes": 2136064392, "md5": "a00bfee46937135c314c944cfd3b70f8"},
        ],
    },

    "zerowaste_s": {
        "tier": 3,
        "nombre": "ZeroWaste-s — partes sintéticas (multiparte)",
        "dominio": "Cinta transportadora · sintético",
        "contenido": "ZIP multiparte .z01-.z03 + .zip",
        "licencia": "CC BY-NC 4.0",
        "cita": "Bashkirova et al., CVPR 2022",
        "archivos": [
            {"nombre": "zerowaste-s-parts.z01", "url": ZEN.format(rec=6412647, key="zerowaste-s-parts.z01"),
             "bytes": 3221225472, "md5": "7d39e260b7f6b4ee7a9b078c923176cc"},
            {"nombre": "zerowaste-s-parts.z02", "url": ZEN.format(rec=6412647, key="zerowaste-s-parts.z02"),
             "bytes": 3221225472, "md5": "47065ff56e2eb895c78587636f0408d2"},
            {"nombre": "zerowaste-s-parts.z03", "url": ZEN.format(rec=6412647, key="zerowaste-s-parts.z03"),
             "bytes": 3221225472, "md5": "d7287fd9dcbf9ade985e5d8fe3850800"},
            {"nombre": "zerowaste-s-parts.zip", "url": ZEN.format(rec=6412647, key="zerowaste-s-parts.zip"),
             "bytes": 966339053, "md5": "10a38443fe43eedad4fecadbcbe86589"},
        ],
    },
}

_tiers_on = {1: INCLUIR_TIER1, 2: INCLUIR_TIER2, 3: INCLUIR_TIER3}
SELECCION = {k: v for k, v in DATASETS.items() if _tiers_on.get(v["tier"], False)}

print("{:<18} {:<3} {:>12}   {}".format("DATASET", "T", "TAMAÑO", "DOMINIO"))
print("─" * 88)
_tot = 0
for k, v in SELECCION.items():
    b = sum(a["bytes"] or 0 for a in v["archivos"])
    _tot += b
    print("{:<18} {:<3} {:>12}   {}".format(k, v["tier"], _fmt(b) if b else "variable", v["dominio"]))
print("─" * 88)
print("{:<18} {:<3} {:>12}".format("TOTAL declarado", "", _fmt(_tot)))
print("\nRoLID-11K y los archivos de GitHub no declaran tamaño; se resuelven al descargar.")

## 3 · Preflight - disco, herramientas y red

In [ ]:
_ok = True

# ── Espacio en disco ─────────────────────────────────────────────────────────
_us = shutil.disk_usage(DATA_RAW)
_necesario = sum(a["bytes"] or 0 for v in SELECCION.values() for a in v["archivos"])
_necesario += 8 * 1024**3    # margen para RoLID (~5.2 GB) y archivos sin tamaño declarado
_necesario *= 2              # bronze duplica aproximadamente el volumen de raw

print("Disco destino  : {}".format(DATA_RAW))
print("  libre        : {}".format(_fmt(_us.free)))
print("  necesario ~  : {}   (raw + bronze)".format(_fmt(_necesario)))
if _us.free < _necesario:
    print("  [ AVISO ] espacio justo o insuficiente — desactiva TIER2/TIER3 o libera disco")
    _ok = False
else:
    print("  holgura      : {}".format(_fmt(_us.free - _necesario)))

# ── Herramientas ─────────────────────────────────────────────────────────────
print("\nHerramientas:")
_curl = shutil.which("curl")
print("  curl         : {}".format(_curl or "NO ENCONTRADO"))
_ok = _ok and bool(_curl)

try:
    import gdown
    print("  gdown        : {}".format(gdown.__version__))
except ImportError:
    print("  gdown        : falta — instalando ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=False)
    try:
        import gdown
        print("  gdown        : {} (instalado)".format(gdown.__version__))
    except ImportError:
        print("  gdown        : instalación falló — RoLID-11K deberá bajarse a mano")

print("  caffeinate   : {}".format(shutil.which("caffeinate") or "no disponible (solo macOS)"))

# ── Conectividad ─────────────────────────────────────────────────────────────
print("\nConectividad:")
for _n, _u in (("Zenodo", "https://zenodo.org"),
               ("GitHub raw", "https://raw.githubusercontent.com"),
               ("Google Drive", "https://drive.google.com")):
    _r = subprocess.run(["curl", "-s", "-o", "/dev/null", "-w", "%{http_code} en %{time_total}s",
                         "--max-time", "20", _u], capture_output=True, text=True)
    print("  {:<13}: {}".format(_n, _r.stdout.strip()))

print("\n" + ("Listo para descargar." if _ok else "Revisa los avisos antes de continuar."))

## 4 · Anti-suspensión

`caffeinate` impide que macOS suspenda el sistema, el disco o la red mientras dure el proceso.
La pantalla sí puede apagarse.

In [ ]:
_caf = None
if shutil.which("caffeinate"):
    # -d pantalla · -i sistema · -m disco · -s con corriente · -u simula actividad
    _caf = subprocess.Popen(["caffeinate", "-dimsu"])
    print("caffeinate activo (PID {}) — la Mac no se suspenderá.".format(_caf.pid))
    print("Se detiene solo en la celda 11. Si interrumpes el notebook, ciérralo con:")
    print("    kill {}".format(_caf.pid))
else:
    print("caffeinate no disponible. Desactiva la suspensión manualmente.")

## 5 · Motor de descarga

In [ ]:
from IPython.display import clear_output

MARCA = {"pendiente": " ", "trabajando": "»", "ok": "✓", "error": "✗"}


def _md5(path, chunk=8 * 1024 * 1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for blk in iter(lambda: f.read(chunk), b""):
            h.update(blk)
    return h.hexdigest()


def _tamano_dir(p):
    try:
        return sum(f.stat().st_size for f in Path(p).rglob("*") if f.is_file())
    except OSError:
        return 0


def _contenido_valido(p):
    """Verifica un archivo por su contenido. Es la única referencia fiable cuando
    la fuente no publica ni tamaño ni MD5 (caso de GitHub raw)."""
    if not p.exists() or p.stat().st_size == 0:
        return False
    suf = p.suffix.lower()
    try:
        if suf == ".json":
            with open(p) as f:
                json.load(f)
        elif suf == ".csv":
            with open(p) as f:
                next(csv.reader(f))
        elif suf == ".zip":
            with zipfile.ZipFile(p) as zf:
                zf.namelist()
        return True
    except Exception:
        return False


def _construir_tareas(seleccion):
    """Aplana el registro a una lista de tareas atómicas (una por archivo)."""
    tareas = []
    for ds_id, ds in seleccion.items():
        destino = DATA_RAW / ds_id
        forzar = ds_id in FORZAR_REDESCARGA
        if forzar and destino.exists():
            shutil.rmtree(destino)          # único borrado, y solo si se pidió explícitamente
        destino.mkdir(parents=True, exist_ok=True)

        if ds.get("modo") == "gdown_folder":
            tareas.append({"ds": ds_id, "nombre": "(carpeta Drive)", "tipo": "gdown",
                           "folder_id": ds["folder_id"], "dir": destino, "dest": destino,
                           "esperado_archivos": ds.get("esperado_archivos", []),
                           "bytes": None, "md5": None, "forzar": forzar,
                           "estado": "pendiente", "detalle": ""})
            continue

        for a in ds["archivos"]:
            tareas.append({"ds": ds_id, "nombre": a["nombre"], "tipo": "curl", "url": a["url"],
                           "dir": destino, "dest": destino / a["nombre"],
                           "bytes": a["bytes"], "md5": a["md5"], "forzar": forzar,
                           "estado": "pendiente", "detalle": ""})
    return tareas


def _descargar(t):
    """Descarga una tarea. Reanudable e idempotente."""
    log = LOGS_DIR / "{}__{}.log".format(t["ds"], t["nombre"].replace("/", "_").replace(" ", "_"))

    try:
        # ── Omitir lo que ya está completo ────────────────────────────────────
        # Dos criterios según lo que publique la fuente:
        #   · tamaño declarado (Zenodo)  -> comparación exacta de bytes
        #   · sin referencia (GitHub)    -> validación por contenido
        # En ningún caso se vuelve a descargar algo ya válido.
        if t["tipo"] == "curl" and not t["forzar"] and t["dest"].exists():
            if t["bytes"]:
                if t["dest"].stat().st_size == t["bytes"]:
                    t["estado"] = "ok"
                    t["detalle"] = "ya completo"
                    return t
            elif _contenido_valido(t["dest"]):
                t["estado"] = "ok"
                t["detalle"] = "ya completo"
                return t

        t["estado"] = "trabajando"

        # ── Carpeta de Google Drive ───────────────────────────────────────────
        if t["tipo"] == "gdown":
            # gdown re-escanea la carpeta remota aunque esté todo bajado, y eso
            # tarda decenas de segundos. Si los archivos esperados ya están y son
            # válidos, se evita la llamada por completo.
            esperados = t.get("esperado_archivos") or []
            if not t["forzar"] and esperados:
                presentes = {p.name: p for p in t["dir"].rglob("*") if p.is_file()}
                if all(n in presentes and _contenido_valido(presentes[n]) for n in esperados):
                    t["estado"] = "ok"
                    t["detalle"] = "ya completo ({} arch.)".format(len(esperados))
                    return t

            import gdown
            antes = _tamano_dir(t["dir"])
            with open(log, "a") as lg:
                lg.write("\n=== gdown folder {} @ {} ===\n".format(t["folder_id"], datetime.datetime.now()))
            gdown.download_folder(id=t["folder_id"], output=str(t["dir"]),
                                  quiet=True, use_cookies=False, remaining_ok=True)
            ahora = _tamano_dir(t["dir"])
            t["estado"] = "ok"
            t["detalle"] = "sin cambios" if ahora == antes else _fmt(ahora)
            return t

        # ── HTTP ──────────────────────────────────────────────────────────────
        cmd = ["curl", "-L", "--fail", "--retry", "8", "--retry-delay", "5",
               "--retry-connrefused", "--connect-timeout", "30"]
        if t["bytes"]:
            cmd += ["-C", "-"]                      # reanudable: tamaño conocido
        elif t["dest"].exists():
            t["dest"].unlink()                      # incompleto o corrupto: se rehace
        cmd += ["-o", str(t["dest"]), t["url"]]

        with open(log, "a") as lg:
            lg.write("\n=== {} @ {} ===\n".format(t["url"], datetime.datetime.now()))
            rc = subprocess.run(cmd, stdout=lg, stderr=lg).returncode

        # curl 33/36: el servidor no acepta reanudar, o el archivo ya está completo
        if rc in (33, 36) and t["dest"].exists() and t["bytes"] and t["dest"].stat().st_size == t["bytes"]:
            rc = 0
        if rc != 0:
            t["estado"] = "error"
            t["detalle"] = "curl rc={} · ver {}".format(rc, log.name)
            return t

        if t["bytes"] and t["dest"].stat().st_size != t["bytes"]:
            t["estado"] = "error"
            t["detalle"] = "tamaño {} != {}".format(_fmt(t["dest"].stat().st_size), _fmt(t["bytes"]))
            return t
        if not t["bytes"] and not _contenido_valido(t["dest"]):
            t["estado"] = "error"
            t["detalle"] = "contenido inválido tras descargar"
            return t

        t["estado"] = "ok"
        t["detalle"] = _fmt(t["dest"].stat().st_size)
        return t

    except Exception as e:
        t["estado"] = "error"
        t["detalle"] = "{}: {}".format(type(e).__name__, e)
        return t


def _pintar_descarga(tareas, t0):
    clear_output(wait=True)
    hecho = sum(1 for t in tareas if t["estado"] in ("ok", "error"))
    print("DESCARGA · {}/{} tareas · {}s transcurridos".format(hecho, len(tareas), int(time.time() - t0)))
    print("═" * 96)
    print("{:<2} {:<16} {:<32} {:>22}   {}".format("", "DATASET", "ARCHIVO", "PROGRESO", "NOTA"))
    print("─" * 96)
    for t in tareas:
        if t["tipo"] == "gdown":
            prog = "{} (sin total)".format(_fmt(_tamano_dir(t["dir"])))
        else:
            act = t["dest"].stat().st_size if t["dest"].exists() else 0
            prog = "{} / {}".format(_fmt(act), _fmt(t["bytes"]))
            if t["bytes"]:
                prog += "  {:5.1f}%".format(act / t["bytes"] * 100)
        print("{:<2} {:<16} {:<32} {:>22}   {}".format(
            MARCA[t["estado"]], t["ds"], t["nombre"][:32], prog, t["detalle"][:24]))
    print("═" * 96)


print("Motor de descarga cargado.")

## 6 · Ejecutar descarga

Celda larga con progreso en vivo. **Reanudable**: si se corta, se reejecuta y continúa
donde quedó. Si todo está ya descargado, termina en segundos marcando *ya completo*.

In [ ]:
TAREAS = _construir_tareas(SELECCION)
_t0 = time.time()

print("Lanzando {} tareas con {} descargas en paralelo...".format(len(TAREAS), MAX_PARALELAS))
time.sleep(1.0)

with ThreadPoolExecutor(max_workers=MAX_PARALELAS) as ex:
    _futs = [ex.submit(_descargar, t) for t in TAREAS]
    while not all(f.done() for f in _futs):
        _pintar_descarga(TAREAS, _t0)
        time.sleep(REFRESCO_SEG)
    _pintar_descarga(TAREAS, _t0)

_dur = time.time() - _t0
print("\nTiempo total : {} min {} s".format(int(_dur // 60), int(_dur % 60)))
print("Volumen raw  : {}".format(_fmt(_tamano_dir(DATA_RAW))))

_errores = [t for t in TAREAS if t["estado"] == "error"]
if _errores:
    print("\n[ AVISO ] {} tarea(s) con error — reejecuta esta celda para reintentar:".format(len(_errores)))
    for t in _errores:
        print("   {} / {} — {}".format(t["ds"], t["nombre"], t["detalle"]))
else:
    print("\nTodas las descargas completaron correctamente.")

## 7 · Verificación de integridad

Tres comprobaciones independientes:

- **MD5** contra los checksums oficiales de Zenodo (solo donde la fuente los publica)
- **ZIP** lectura del directorio central, que detecta truncamiento sin descomprimir
- **JSON / CSV** parseo real del archivo, que es la verificación válida para los
  ficheros de anotaciones servidos por GitHub, donde no existe MD5 de referencia

In [ ]:
print("{:<2} {:<48} {:<12} {:<14} {}".format("", "ARCHIVO", "MD5", "CONTENIDO", "TAMAÑO"))
print("─" * 100)

VERIF = []
for t in TAREAS:
    if t["tipo"] == "gdown":
        for p in sorted(Path(t["dir"]).rglob("*")):
            if p.is_file() and not p.name.startswith("."):
                VERIF.append({"ds": t["ds"], "path": p, "md5_esperado": None})
    elif t["dest"].exists():
        VERIF.append({"ds": t["ds"], "path": t["dest"], "md5_esperado": t["md5"]})

for v in VERIF:
    p = v["path"]
    v["bytes"] = p.stat().st_size
    suf = p.suffix.lower()

    # ── MD5 ───────────────────────────────────────────────────────────────────
    if VERIFICAR_MD5 and v["md5_esperado"]:
        v["md5"] = _md5(p)
        v["md5_ok"] = (v["md5"] == v["md5_esperado"])
        col_md5 = "ok" if v["md5_ok"] else "FALLA"
    else:
        v["md5"] = None
        v["md5_ok"] = None
        col_md5 = "sin ref."

    # ── Contenido ─────────────────────────────────────────────────────────────
    v["cont_ok"] = None
    col_cont = "n/a"
    if suf == ".zip":
        try:
            with zipfile.ZipFile(p) as zf:
                v["entradas"] = len(zf.namelist())
            v["cont_ok"] = True
            col_cont = "{} entradas".format(v["entradas"])
        except Exception as e:
            v["cont_ok"] = False
            col_cont = "ZIP corrupto"
            v["error"] = str(e)
    elif suf == ".json":
        try:
            with open(p) as f:
                d = json.load(f)
            v["cont_ok"] = True
            v["entradas"] = len(d.get("images", d)) if isinstance(d, dict) else len(d)
            col_cont = ("{} imgs".format(len(d["images"]))
                        if isinstance(d, dict) and "images" in d else "JSON ok")
        except Exception as e:
            v["cont_ok"] = False
            col_cont = "JSON inválido"
            v["error"] = str(e)
    elif suf == ".csv":
        try:
            with open(p) as f:
                n = sum(1 for _ in csv.reader(f))
            v["cont_ok"] = True
            v["entradas"] = n
            col_cont = "{} filas".format(n)
        except Exception as e:
            v["cont_ok"] = False
            col_cont = "CSV inválido"
            v["error"] = str(e)
    elif suf in (".z01", ".z02", ".z03"):
        col_cont = "parte multi-zip"

    marca = "✗" if (v["md5_ok"] is False or v["cont_ok"] is False) else "✓"
    print("{:<2} {:<48} {:<12} {:<14} {}".format(
        marca, "{}/{}".format(v["ds"], p.name)[:48], col_md5, col_cont, _fmt(v["bytes"])))

print("─" * 100)
_malos = [v for v in VERIF if v["md5_ok"] is False or v["cont_ok"] is False]
if _malos:
    print("[ AVISO ] {} archivo(s) con problema. Bórralos y reejecuta la celda 6:".format(len(_malos)))
    for v in _malos:
        print("   rm '{}'".format(v["path"]))
else:
    print("{} archivo(s) verificados · {} en total".format(
        len(VERIF), _fmt(sum(v["bytes"] for v in VERIF))))

## 8 · Descompresión a `bronze`

`bronze` es `raw` descomprimido y navegable, **sin transformación semántica**. Es la capa
donde se puede abrir una carpeta y mirar las imágenes, que es como realmente se depura
visión por computador.

Estructura homogénea para todos los datasets:

```
bronze/<dataset>/
├── images/        imágenes, con la jerarquía que exijan sus anotaciones
├── masks/         máscaras, solo cuando el dataset las trae
└── annotations/   JSON / CSV originales
```

Reglas aplicadas al extraer:

- Se descarta el ruido de empaquetado: `.git/`, `__MACOSX/`, `.DS_Store`, `.ipynb_checkpoints/`
- **TACO** conserva la jerarquía `batch_N/` porque sus anotaciones referencian `batch_N/xxx.jpg`
- **RoLID** se aplana, porque sus anotaciones referencian solo el nombre del archivo. La carpeta
  de origen (`0`–`23`) es un **índice de sesión** (no la hora del día: la hora real va en el
  timestamp del nombre de archivo) y se preserva aparte en `annotations/carpeta_origen.csv`
  por trazabilidad con el release
- **ZeroWaste-f** conserva la separación oficial `train/val/test` y separa `images/` de `masks/`

In [ ]:
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
RUIDO   = ("/.git/", "__MACOSX", ".DS_Store", ".ipynb_checkpoints", "/._")


def _es_ruido(n):
    return any(r in n for r in RUIDO) or n.endswith("/")


# ── Mapeadores: ruta dentro del ZIP -> ruta relativa dentro de bronze/<ds>/ ───
# Devolver None descarta la entrada.

def _map_taco(n):
    if _es_ruido(n):
        return None
    if "/data/batch_" in n and Path(n).suffix.lower() in IMG_EXT:
        return "images/" + n.split("/data/", 1)[1]        # -> images/batch_1/000006.jpg
    return None


def _map_rolid(n):
    if _es_ruido(n) or Path(n).suffix.lower() not in IMG_EXT:
        return None
    return "images/" + Path(n).name                        # aplanado


def _map_uavvaste(n):
    if _es_ruido(n):
        return None
    if n.startswith("images/") and Path(n).suffix.lower() in IMG_EXT:
        return "images/" + Path(n).name
    if n.startswith("annotations/"):
        return "annotations/" + Path(n).name
    return None


def _map_zerowaste_f(n):
    # Dos profundidades conviven en este ZIP:
    #   splits_final_deblurred/<split>/<data|sem_seg>/<img>   -> 4 niveles
    #   splits_final_deblurred/<split>/labels.json            -> 3 niveles
    if _es_ruido(n):
        return None
    partes = n.split("/")
    if len(partes) == 3 and partes[2].endswith(".json"):
        return "annotations/{}_{}".format(partes[1], partes[2])
    if len(partes) < 4:
        return None
    split, tipo, archivo = partes[1], partes[2], partes[-1]
    if Path(archivo).suffix.lower() in IMG_EXT:
        if tipo == "data":
            return "images/{}/{}".format(split, archivo)
        if tipo == "sem_seg":
            return "masks/{}/{}".format(split, archivo)
    if archivo.endswith(".json"):
        return "annotations/{}_{}".format(split, archivo)
    return None


def _map_zerowaste_w(n):
    if _es_ruido(n) or Path(n).suffix.lower() not in IMG_EXT:
        return None
    return "images/" + n                                   # conserva before/ y after/


BRONZE = {
    "taco":        {"zips": ["TACO.zip"], "map": _map_taco,
                    "copiar": ["annotations.json", "annotations_unofficial.json", "all_image_urls.csv"],
                    "esperado": 1500},
    "rolid11k":    {"zips": ["0-11.zip", "12_15.zip", "15_18.zip", "18_21.zip", "21_24.zip"],
                    "map": _map_rolid,
                    "copiar": ["training.json", "validation.json", "testing.json"],
                    "esperado": 11564},
    "uavvaste":    {"zips": ["UAVVasteDataset.zip"], "map": _map_uavvaste,
                    "copiar": [], "esperado": 772},
    "detectwaste_ann": {"zips": [], "map": None,
                    "copiar": "*.json", "esperado": 0,
                    "nota": "solo anotaciones, sin imágenes"},
    "zerowaste_f": {"zips": ["zerowaste-f-final.zip"], "map": _map_zerowaste_f,
                    "copiar": [], "esperado": 9006,
                    "nota": "fuera del alcance: dominio industrial"},
    "zerowaste_w": {"zips": ["zerowaste-w.zip"], "map": _map_zerowaste_w,
                    "copiar": [], "esperado": 2410,
                    "nota": "ZIP cifrado, sin contraseña publicada"},
}


def _extraer(tarea):
    """Descomprime un dataset de raw a bronze. Idempotente."""
    ds   = tarea["ds"]
    cfg  = BRONZE[ds]
    orig = DATA_RAW / ds
    dest = DATA_BRONZE / ds
    marca = dest / "_bronze_ok.json"

    try:
        # ── Rehacer desde cero solo si se pidió explícitamente ────────────────
        if ds in FORZAR_REEXTRACCION and dest.exists():
            shutil.rmtree(dest)

        tarea["estado"] = "trabajando"
        dest.mkdir(parents=True, exist_ok=True)

        # ── Planificar miembros de todos los ZIP ──────────────────────────────
        # El plan se recalcula siempre. Es barato (solo lee el directorio central
        # del ZIP) y hace la extracción auto-reparable: si una versión anterior
        # dejó archivos fuera, esta ejecución los completa sin borrar nada.
        plan = []
        for zn in cfg["zips"]:
            zp = orig / zn
            if not zp.exists():
                continue
            with zipfile.ZipFile(zp) as zf:
                # bit 0 de flag_bits = contenido cifrado. Sin contraseña no hay nada
                # que hacer, y conviene decirlo con claridad en vez de reventar.
                if any(i.flag_bits & 0x1 for i in zf.infolist()):
                    tarea["estado"] = "error"
                    tarea["detalle"] = "ZIP cifrado, se requiere contraseña"
                    return tarea
                for n in zf.namelist():
                    rel = cfg["map"](n) if cfg["map"] else None
                    if rel:
                        plan.append((zp, n, rel))
        tarea["total"] = len(plan) or 1

        # ── Camino rápido: nada que hacer ─────────────────────────────────────
        pendientes = [p for p in plan if not (dest / p[2]).exists()]
        nombres_ann = (sorted(p.name for p in orig.glob("*.json"))
                       if cfg["copiar"] == "*.json" else cfg["copiar"])
        ann_faltan = [n for n in nombres_ann if not (dest / "annotations" / n).exists()]
        if not pendientes and not ann_faltan and marca.exists():
            tarea["hechos"] = tarea["total"]
            tarea["estado"] = "ok"
            tarea["detalle"] = "ya extraído"
            return tarea
        tarea["hechos"] = len(plan) - len(pendientes)

        # ── Extraer ───────────────────────────────────────────────────────────
        zp_actual, zf = None, None
        try:
            for zp, n, rel in pendientes:
                if zp != zp_actual:
                    if zf:
                        zf.close()
                    zf = zipfile.ZipFile(zp)
                    zp_actual = zp
                salida = dest / rel
                salida.parent.mkdir(parents=True, exist_ok=True)
                with zf.open(n) as src, open(salida, "wb") as dst:
                    shutil.copyfileobj(src, dst, 1024 * 1024)
                tarea["hechos"] += 1
        finally:
            if zf:
                zf.close()

        # ── Copiar anotaciones sueltas ────────────────────────────────────────
        adir = dest / "annotations"
        for nom in nombres_ann:
            src = orig / nom
            if src.exists():
                adir.mkdir(parents=True, exist_ok=True)
                shutil.copy2(src, adir / nom)

        # ── Metadato de carpeta de origen (RoLID) ─────────────────────────────
        # La carpeta dentro del ZIP (0-23) es un índice de sesión, NO la hora del
        # día (verificado contra el timestamp del nombre: solo 64 de 11.564
        # coinciden). La hora real y el video van en el propio nombre de archivo:
        # 20220605231413_000097_Trim_frame1.jpg -> 2022-06-05 23:14:13. Se
        # preserva la carpeta igualmente por trazabilidad con el release.
        franja = [(Path(n).name, n.split("/")[0]) for _, n, _ in plan] if ds == "rolid11k" else []
        if ds == "rolid11k" and franja:
            adir.mkdir(parents=True, exist_ok=True)
            with open(adir / "carpeta_origen.csv", "w", newline="") as f:
                w = csv.writer(f)
                w.writerow(["file_name", "carpeta_origen"])
                w.writerows(sorted(set(franja)))

        n_final = sum(1 for p in dest.rglob("*") if p.is_file() and p.name != "_bronze_ok.json")
        marca.write_text(json.dumps({
            "extraido": datetime.datetime.now().isoformat(timespec="seconds"),
            "archivos": n_final,
            "imagenes": sum(1 for p in (dest / "images").rglob("*")
                            if p.is_file() and p.suffix.lower() in IMG_EXT)
            if (dest / "images").exists() else 0,
        }, indent=2))

        tarea["hechos"] = tarea["total"]
        tarea["estado"] = "ok"
        tarea["detalle"] = _fmt(_tamano_dir(dest))
        return tarea

    except Exception as e:
        tarea["estado"] = "error"
        tarea["detalle"] = "{}: {}".format(type(e).__name__, e)
        return tarea


def _pintar_extraccion(tareas, t0):
    clear_output(wait=True)
    hecho = sum(1 for t in tareas if t["estado"] in ("ok", "error"))
    print("DESCOMPRESIÓN raw -> bronze · {}/{} datasets · {}s".format(
        hecho, len(tareas), int(time.time() - t0)))
    print("═" * 88)
    print("{:<2} {:<18} {:>26}   {}".format("", "DATASET", "PROGRESO", "NOTA"))
    print("─" * 88)
    for t in tareas:
        tot = t["total"] or 1
        pct = t["hechos"] / tot * 100 if tot else 0
        print("{:<2} {:<18} {:>12,} / {:>9,}  {:5.1f}%   {}".format(
            MARCA[t["estado"]], t["ds"], t["hechos"], t["total"], pct, t["detalle"][:26]))
    print("═" * 88)


print("Motor de descompresión cargado.")

## 9 · Ejecutar descompresión

Idempotente: cada dataset deja una marca `_bronze_ok.json` y se omite en ejecuciones
posteriores. Para rehacer uno, borrar su carpeta en `bronze/`.

In [ ]:
_libre = shutil.disk_usage(DATA_BRONZE).free
print("Espacio libre: {}   ·   estimado necesario: ~22 GB".format(_fmt(_libre)))
if _libre < 25 * 1024**3:
    print("[ AVISO ] espacio ajustado para la descompresión.")

TAREAS_EXT = [{"ds": k, "estado": "pendiente", "hechos": 0, "total": 0, "detalle": ""}
              for k in SELECCION if k in BRONZE]
_t0 = time.time()

print("Descomprimiendo {} datasets con {} en paralelo...".format(len(TAREAS_EXT), MAX_EXTRACCION))
time.sleep(1.0)

with ThreadPoolExecutor(max_workers=MAX_EXTRACCION) as ex:
    _futs = [ex.submit(_extraer, t) for t in TAREAS_EXT]
    while not all(f.done() for f in _futs):
        _pintar_extraccion(TAREAS_EXT, _t0)
        time.sleep(REFRESCO_SEG)
    _pintar_extraccion(TAREAS_EXT, _t0)

_dur = time.time() - _t0
print("\nTiempo total   : {} min {} s".format(int(_dur // 60), int(_dur % 60)))
print("Volumen bronze : {}".format(_fmt(_tamano_dir(DATA_BRONZE))))

_err = [t for t in TAREAS_EXT if t["estado"] == "error"]
if _err:
    print("\n[ AVISO ] {} dataset(s) con error:".format(len(_err)))
    for t in _err:
        print("   {} — {}".format(t["ds"], t["detalle"]))
else:
    print("\nDescompresión completa.")

## 10 · Verificación de `bronze`

Contraste del conteo real de imágenes extraídas contra lo esperado según cada publicación
original. Cualquier desviación aquí indica un problema de extracción, no de descarga.

In [ ]:
print("{:<2} {:<17} {:>9} {:>9} {:>9} {:>7} {:>10}   {}".format(
    "", "DATASET", "IMÁGENES", "ESPERADO", "MÁSCARAS", "ANOTAC.", "TAMAÑO", "NOTA"))
print("─" * 108)

RESUMEN = []
for k in SELECCION:
    if k not in BRONZE:
        continue
    cfg = BRONZE[k]
    d = DATA_BRONZE / k
    n_img = sum(1 for p in (d / "images").rglob("*")
                if p.is_file() and p.suffix.lower() in IMG_EXT) if (d / "images").exists() else 0
    n_msk = sum(1 for p in (d / "masks").rglob("*")
                if p.is_file() and p.suffix.lower() in IMG_EXT) if (d / "masks").exists() else 0
    n_ann = sum(1 for p in (d / "annotations").glob("*")
                if p.is_file()) if (d / "annotations").exists() else 0

    # zerowaste_f declara imágenes + máscaras juntas dentro del ZIP
    esp_img = cfg["esperado"] // 2 if k == "zerowaste_f" else cfg["esperado"]
    solo_ann = not cfg["zips"]
    ok = True if solo_ann else (n_img == esp_img)

    print("{:<2} {:<17} {:>9} {:>9} {:>9} {:>7,} {:>10}   {}".format(
        "✓" if ok else "✗", k,
        "n/a" if solo_ann else format(n_img, ","),
        "n/a" if solo_ann else format(esp_img, ","),
        "n/a" if n_msk == 0 else format(n_msk, ","),
        n_ann, _fmt(_tamano_dir(d)), cfg.get("nota", "")))

    RESUMEN.append({"dataset": k, "imagenes": n_img, "esperado": esp_img,
                    "mascaras": n_msk, "anotaciones": n_ann, "solo_anotaciones": solo_ann,
                    "bytes": _tamano_dir(d), "ok": bool(ok), "nota": cfg.get("nota", "")})

print("─" * 108)
print("{:<2} {:<17} {:>9,} {:>9} {:>9,} {:>7,} {:>10}".format(
    "", "TOTAL", sum(r["imagenes"] for r in RESUMEN), "",
    sum(r["mascaras"] for r in RESUMEN), sum(r["anotaciones"] for r in RESUMEN),
    _fmt(_tamano_dir(DATA_BRONZE))))

_mal = [r for r in RESUMEN if not r["ok"]]
if _mal:
    print("\n[ AVISO ] no extraídos: " + ", ".join(r["dataset"] for r in _mal))
    for r in _mal:
        print("   {} — {}".format(r["dataset"], r["nota"] or "revisar log"))
    print("\nUn dataset marcado como cifrado o fuera de alcance no bloquea el proyecto:")
    print("queda en raw/ y se documenta su exclusión en el EDA y en la curación a silver.")
else:
    print("\nTodos los conteos coinciden con las publicaciones originales.")

## 11 · Manifiesto y cierre

In [ ]:
manifiesto = {
    "generado": datetime.datetime.now().isoformat(timespec="seconds"),
    "host": os.uname().nodename,
    "rutas": {"raw": str(DATA_RAW), "bronze": str(DATA_BRONZE)},
    "tiers": {"tier1": INCLUIR_TIER1, "tier2": INCLUIR_TIER2, "tier3": INCLUIR_TIER3},
    "datasets": {
        k: {c: v[c] for c in ("nombre", "tier", "dominio", "contenido", "licencia", "cita")}
        for k, v in SELECCION.items()
    },
    "raw_archivos": [
        {"dataset": v["ds"], "ruta": str(v["path"].relative_to(DATA_RAW)), "bytes": v["bytes"],
         "md5": v["md5"], "md5_ok": v["md5_ok"], "contenido_ok": v["cont_ok"],
         "entradas": v.get("entradas")}
        for v in VERIF
    ],
    "bronze": RESUMEN,
    "raw_bytes": sum(v["bytes"] for v in VERIF),
    "bronze_bytes": sum(r["bytes"] for r in RESUMEN),
}
MANIFEST.write_text(json.dumps(manifiesto, indent=2, ensure_ascii=False))
print("Manifiesto -> {}".format(MANIFEST))

if _caf and _caf.poll() is None:
    _caf.terminate()
    print("caffeinate detenido (PID {}). La Mac puede volver a suspenderse.".format(_caf.pid))

print("\n{:<14} {:>14}".format("CAPA", "TAMAÑO"))
print("─" * 30)
print("{:<14} {:>14}".format("raw", _fmt(manifiesto["raw_bytes"])))
print("{:<14} {:>14}".format("bronze", _fmt(manifiesto["bronze_bytes"])))
print("─" * 30)
print("{:<14} {:>14}".format("TOTAL", _fmt(manifiesto["raw_bytes"] + manifiesto["bronze_bytes"])))
print("\nEspacio libre restante: {}".format(_fmt(shutil.disk_usage(DATA_RAW).free)))
print("\nSiguiente paso: notebook 01 — EDA sobre bronze.")

## 12 · Notas

### Hallazgos ya confirmados sobre estos datos

| Dataset | Observación |
|---|---|
| **RoLID-11K** | La imagen `20220606234554_000167_Trim_1_frame1168.jpg` aparece **en `validation` y en `testing`** a la vez (ids 969 y 1520). Fuga en el release oficial. Se corrige en `silver`, retirándola de `validation` para no alterar el test y poder comparar contra los baselines publicados |
| **RoLID-11K** | Declara la categoría `0: 'None'`, que **no tiene ninguna anotación**. Es un placeholder del exportador VIA. Al convertir hay que mapear explícitamente para no crear una clase fantasma |
| **RoLID-11K** | Todas las imágenes son 1920x1080, no 4K. Distribución de tamaño de objeto verificada contra la Figura 4 del paper: 83.7 / 16.0 / 0.3 % (train) |
| **RoLID-11K** | **Los splits oficiales tienen fuga a nivel de video**: 22 de 84 videos reparten sus frames en más de un split y el 58,2 % de las imágenes de test comparte video con train. Los baselines publicados están inflados por fuga de escena; en silver se añade un split propio agrupado por video |
| **TACO** | El nombre base se repite entre batches (`000006.jpg` existe en 9): toda unión debe usar la ruta completa `batch_N/archivo`. Además ~37 % de las imágenes llevan rotación EXIF pendiente: las cajas están anotadas sobre la imagen ya girada |
| **TACO** | `annotations_unofficial.json` referencia **3.831 imágenes**, de las cuales **ninguna** viene en `TACO.zip`. Sus URLs sí están en `all_image_urls.csv`. Aprovecharlas exige descargar de Flickr — decisión pendiente para el EDA |
| **TACO** | El ZIP incluye el repositorio git completo (373 entradas en `.git/`) y 14 *resource forks* de macOS (`__MACOSX/.../._xxx.jpg`, 177 bytes cada uno) que un conteo ingenuo confunde con imágenes: parecen 1.514 pero son **1.500 reales**, exactamente el conjunto anotado. Todo ese ruido se descarta al pasar a bronze |
| **ZeroWaste-w** | `zerowaste-w.zip` **está cifrado con contraseña**. Se descarga entero y su MD5 es correcto, pero no se puede extraer. La contraseña no está publicada ni en el repo `dbash/zerowaste` ni en Zenodo. Se deja en `raw/` y se documenta la exclusión |
| **ZeroWaste-f** | Dentro del ZIP conviven dos profundidades de ruta: las imágenes a 4 niveles y los `labels.json` a 3. Un mapeador que asuma profundidad fija pierde silenciosamente las anotaciones |
| **Alcance** | ZeroWaste (f y w) queda **fuera del alcance de modelado**: es clasificación de residuos en cinta transportadora industrial, sin contexto de escena urbana. Se conserva descargado y se caracteriza en el EDA para justificar la exclusión con datos, no por criterio |

### Control de re-ejecución

La notebook **nunca borra nada por su cuenta**: lo ya presente y válido se omite. Para rehacer
algo hay que nombrarlo explícitamente en la celda 1:

```python
FORZAR_REDESCARGA   = ["taco"]         # borra raw/taco/ y vuelve a bajarlo
FORZAR_REEXTRACCION = ["zerowaste_f"]  # borra bronze/zerowaste_f/ y reextrae
```

Criterios de omisión, según lo que publique cada fuente:

| Fuente | Referencia disponible | Criterio |
|---|---|---|
| Zenodo | tamaño exacto + MD5 | comparación de bytes |
| GitHub raw | ninguna | validación por contenido (parseo real del JSON/CSV) |
| Google Drive | ninguna | presencia y validez de los archivos esperados, evitando el re-escaneo remoto |

### Fuentes que no se descargan aquí

| Fuente | Motivo | Acción |
|---|---|---|
| **StreetView-Waste** | Los 4 ZIP devuelven **HTTP 401** pese a anunciarse como público | Escribir a `diogo.paulo@ubi.pt` |
| **Imágenes de TACO vía Flickr** | `download.py` baja de a una desde Flickr: lento y con enlaces caídos | Innecesario para las 1.500 oficiales; pendiente para las no oficiales |
| **OpenLitterMap** | Solo etiquetas geolocalizadas, sin bounding boxes | Su índice llega en `detectwaste_ann` |
| **Lima-OOD** | Se captura y anota a mano | Ver `docs/protocolo_captura_lima.md` |

### ZIP multiparte (solo TIER 3)

```bash
zip -s 0 zerowaste-aug.zip --out zerowaste-aug-completo.zip && unzip zerowaste-aug-completo.zip
```

### Rehacer una capa

```bash
rm -rf code/data/bronze/<dataset>      # rehace solo la descompresión
rm -rf code/data/raw/<dataset>         # fuerza la re-descarga completa
```